# lmsyz_genai_ie_rfs: 60-second Colab quickstart

Run a prompt over a 3-row pandas DataFrame and get a structured DataFrame
back. Companion code to Li, Mai, Shen, Yang, Zhang (2026), *RFS*
39(1):253–296.

For the full workshop notebook (Glassdoor corpus, schema, batch API,
OpenRouter), see `notebooks/01_quickstart.ipynb` in this repo.

## 1. Install

In [ ]:
!pip install -q -U lmsyz_genai_ie_rfs

## 2. Set your OpenAI key

In [ ]:
import os

# os.environ["OPENAI_API_KEY"] = "sk-..."   # uncomment and paste

## 3. Build the input DataFrame

In [ ]:
import pandas as pd

df = pd.DataFrame(
    {
        "id": [1, 2, 3],
        "text": [
            "Apple CEO Tim Cook announced the iPhone 17 at WWDC in June 2025.",
            "Tesla's $2.6 billion acquisition of SolarCity in 2016 enabled its entry into the solar market.",
            "Pfizer's decision to spin off its consumer health unit was driven by activist pressure from Trian Partners.",
        ],
    }
)
df

## 4. Write the extraction prompt

In [ ]:
prompt = """
You are an information-extraction assistant. For each input row, analyze the text and extract structured information.

Step-by-step instructions:

1. input_id: Copy the input_id from the row verbatim.
2. entities: List every named entity mentioned in the text. For each entity give:
   - name: the surface form as it appears in the text.
   - type: one of "PERSON", "ORG", "PRODUCT", "DATE", "MONEY".
3. causal_triples: If the text explicitly states a cause and effect, list each as a
   three-element array ["cause", "relation", "effect"]. If there is no explicit
   causation, return an empty list []. All elements should be concisely summarized, in three words or less.
4. sentiment: One of "positive", "neutral", or "negative".

Return a JSON object with this EXACT structure:

{
  "all_results": [
    {
      "input_id": "1",
      "entities": [
        {"name": "Apple",    "type": "ORG"},
        {"name": "Tim Cook", "type": "PERSON"}
      ],
      "causal_triples": [[cause_1, relation_1, effect_1], [cause_2, relation_2, effect_2], ...],
      "sentiment": "positive/neutral/negative"
    }
  ]
}

Do not include any fields besides input_id, entities, causal_triples, and sentiment.
"""

## 5. Run the extraction

In [ ]:
from lmsyz_genai_ie_rfs import extract_df

out = extract_df(
    df,
    prompt=prompt,
    backend="openai",
    model="gpt-4.1-mini",
    cache_path="demo.sqlite",
    chunk_size=5,
    max_workers=20,
    id_col="id",
    text_col="text",
    api_key=os.environ["OPENAI_API_KEY"],
)
out

## 6. Explode the entities column

Each row's `entities` field is a list of `{name, type}` dicts. Use
`df.explode` to turn each list into multiple rows (one entity per row).

In [ ]:
flat = out.explode("entities", ignore_index=True)
flat

Then `pd.json_normalize` flattens each dict's keys into separate columns.

In [ ]:
flat = pd.concat(
    [flat.drop(columns="entities"), pd.json_normalize(flat["entities"])],
    axis=1,
)
flat